# Results notebook: Distributed Training and Its Carbon Footprint

Loads `results/runs.jsonl` (raw results from `scripts/03_run_experiments.py`), regenerates every table/plot, and states the sizing recommendation. **Run `python scripts/04_analyze.py` first**, or run the cell below.

> If you are looking at smoke-test/synthetic numbers, they are for pipeline testing only and must not be reported.

In [ ]:
import os
if not os.path.isdir('scripts'):      # works when launched from notebooks/ or repo root
    os.chdir('..')
import subprocess, sys, json, pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
RUNS = 'results/runs.jsonl'
subprocess.run([sys.executable, 'scripts/04_analyze.py', '--runs', RUNS, '--outdir', 'results'], check=True, capture_output=True)
raw = pd.DataFrame([json.loads(l) for l in open(RUNS) if l.strip()])
print(len(raw), 'runs |', raw.rows_train.max(), 'max train rows | host CPUs:', raw.host_cpus.iloc[0])
raw.groupby(['mode','cores']).size().unstack(0)

## 1. Experimental setup
Hardware, Spark version, data size and hyper-parameters (edit the markdown below with your real setup).

In [ ]:
raw[['spark','platform','hash_bits','max_iter','price_per_vcpu_hour']].drop_duplicates().T

## 2. Strong scaling (fixed data, more cores)

In [ ]:
S = pd.read_csv('results/summary_strong.csv')
S[['cores','fit_s','total_s','speedup_fit','efficiency','cost_usd','energy_kwh','co2e_kg','auc','n_runs']]

In [ ]:
Image('results/strong_scaling.png')

## 3. Weak scaling (data grows with cores)

In [ ]:
W = pd.read_csv('results/summary_weak.csv')
W[['cores','rows_train','rows_per_core','fit_s','efficiency','co2e_kg']]

In [ ]:
Image('results/weak_scaling.png')

## 4. Cost, energy and carbon

In [ ]:
Image('results/cost_carbon.png')

In [ ]:
Image('results/tradeoff.png')

## 5. Where do extra machines stop being worthwhile?
Marginal time saved vs extra carbon and cost for each step up in cluster size.

In [ ]:
pd.read_csv('results/marginal_analysis.csv')

In [ ]:
display(Markdown(open('results/recommendation.md').read()))

## 6. Quality guard (Responsible AI)
Does scaling change model quality overall or on the `I1 missing` slice?

In [ ]:
Image('results/slice_auc.png')

## 7. Run-to-run variability
Std-dev of repeated runs. If it is large relative to the differences you claim, say so in the report.

In [ ]:
S[['cores','fit_s','fit_s_std','total_s','total_s_std','energy_kwh','energy_kwh_std']]

## 8. Conclusions
*Write 3-5 bullet points here with your own numbers: best size, diminishing-returns point, cost/carbon at that point, caveats.*